# Code tour — where everything actually lives

Every cell below **prints the real source** out of `bruisekit/`, read from disk
at the moment you run it. Nothing here is a copy: if this notebook shows it, that
is the code that trains.

Each block prints a `file:line` header. In Jupyter and VS Code that is a
**clickable link** — click it to open the file at that line.

Run the setup cell once, then jump to whichever question you need. The order is
roughly the order these questions get asked: data → metric → loss → distillation
→ foundation models → training → thresholding → analysis → the guards.

In [ ]:
import inspect
import re
import textwrap
from pathlib import Path

try:
    from IPython.display import Code, Markdown, display
except ImportError:                    # plain python, or a kernel without IPython
    Code = Markdown = lambda x, language=None: x
    display = print

# The bundle root, found from wherever this notebook was opened.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "bruisekit").is_dir()), Path.cwd())
print(f"reading source from  {ROOT / 'bruisekit'}")


def _block(lines, start):
    """The def/class beginning at `start` (1-indexed), to the end of its body."""
    head = lines[start - 1]
    indent = len(head) - len(head.lstrip())
    out = [head]
    for ln in lines[start:]:
        if ln.strip() and (len(ln) - len(ln.lstrip())) <= indent:
            break
        out.append(ln)
    while out and not out[-1].strip():
        out.pop()
    return out


def show(rel, anchor, body=True, max_lines=90):
    """Print the real implementation of `anchor` in `rel`, with line numbers.

    `anchor` is a regex matched against whole lines; the first match wins. The
    file is re-read on every call, so this cannot go stale relative to the code.
    """
    path = ROOT / rel
    if not path.exists():
        display(Markdown(f"**missing:** `{rel}` — is this the bundle root?"))
        return
    lines = path.read_text(encoding="utf-8").splitlines()
    pat = re.compile(anchor)
    hit = next((i for i, ln in enumerate(lines, 1) if pat.search(ln)), None)
    if hit is None:
        display(Markdown(f"**not found:** `{anchor}` in `{rel}` — the code was "
                         f"renamed. Regenerate this notebook with "
                         f"`scripts/89_generate_code_tour_notebook.py`, which "
                         f"fails loudly on a stale anchor."))
        return

    seg = _block(lines, hit) if body else [lines[hit - 1]]
    truncated = len(seg) > max_lines
    if truncated:
        seg = seg[:max_lines] + [f"    ...  ({len(_block(lines, hit)) - max_lines}"
                                 f" more lines — open the file to read on)"]

    display(Markdown(f"#### `{rel}:{hit}`"))
    width = len(str(hit + len(seg)))
    numbered = "\n".join(f"{hit + k:>{width}} | {ln}" for k, ln in enumerate(seg))
    display(Code(numbered, language="python"))


def docstring(rel, anchor=None):
    """Just the module or function docstring — the WHY, without the body."""
    path = ROOT / rel
    lines = path.read_text(encoding="utf-8").splitlines()
    if anchor is None:
        txt = path.read_text(encoding="utf-8")
        m = re.search(r'"""(.*?)"""', txt, re.S)
        display(Markdown(f"#### `{rel}` — module docstring"))
        display(Markdown("```\n" + (m.group(1).strip() if m else "") + "\n```"))
        return
    show(rel, anchor)


print("ready — `show(file, anchor)` prints source, `docstring(file)` prints the why")

---

### How is the data split, and can a patient appear in both halves?

Splits are grouped by **subject**, never by image. Two photographs of one bruise cannot land on opposite sides of the split — that is the single easiest way to inflate a segmentation result, and the check is re-asserted at build time and again in every notebook.

In [ ]:
show("bruisekit/data.py", r"^class BruiseDataset")
show("bruisekit/data.py", r"def build_augmentation")

### How is accuracy measured?

Dice per image, then averaged — never pooled over the batch. Pooling lets one large bruise dominate. Note the both-empty case scores 1.0, and the sweep in a later cell matches that convention exactly.

In [ ]:
show("bruisekit/metrics.py", r"def dice_np")
show("bruisekit/evaluate.py", r"def evaluate_at_cut")

### What is the loss function?

Dice + BCE. Dice is computed **per image** and then averaged, for the reason in the docstring.

In [ ]:
show("bruisekit/losses.py", r"^class DiceBCELoss")
show("bruisekit/losses.py", r"^class SupervisedLoss")

### How does knowledge distillation work here?

`alpha * DiceBCE(student, ground truth) + (1-alpha) * BCE(student, calibrated teacher probability)`. **Read the module docstring first** — it states explicitly that this is calibrated soft-target distillation and *not* Hinton KD, and why that distinction matters for the write-up.

In [ ]:
show("bruisekit/losses.py", r"None")
show("bruisekit/losses.py", r"^class DistillLoss")

### Why is the teacher calibrated, and how?

An uncalibrated teacher's soft label is the hard label with extra steps. Temperature is fitted by NLL on validation (Guo et al. 2017). The code never falls back to T = 1.

In [ ]:
show("bruisekit/engine.py", r"def calibrate_temperature")

### How do you attach a segmentation head to a model that has none?

Foundation encoders output a grid of features, not a mask. This ~1M-parameter decoder turns a 40×40 feature grid into a full-resolution mask. **Stages N3, N4 and O all import this same class** rather than re-typing it, or their numbers would not be comparable.

In [ ]:
show("bruisekit/finetune_n3.py", r"^class ConvDecodeHead")
show("bruisekit/foundation.py", r"^class LinearProbeHead")

### How do you freeze the encoder?

Two things, and the second is the one people forget: `requires_grad=False` **and** pinning the module to `eval()`. Freezing weights alone still lets BatchNorm statistics and dropout drift.

In [ ]:
show("bruisekit/dermprobe.py", r"def _freeze")

### How many layers did you actually train?

The last **6 of 12** transformer blocks plus the final normalisation layer — about half the encoder — identical for DINOv2, DermLIP, SAM and MedSAM so the comparison is fair. Note the function **raises** if it ends up with zero trainable parameters: an arm that silently trains nothing scores low-to-mid and is indistinguishable from a real result.

In [ ]:
show("bruisekit/samprobe.py", r"^UNFREEZE_BLOCKS = ")
show("bruisekit/finetune_n3.py", r"def unfreeze_last")
show("bruisekit/samprobe.py", r"def unfreeze_last")

### How did you use MedSAM without giving it a prompt?

SAM and MedSAM are promptable — you point at the thing you want. Our pipeline is automatic, so we keep the image encoder and discard the prompt encoder and mask decoder. The class docstring lists the three things SAM does that no other encoder in the study does, each of which would silently corrupt the arm if mishandled.

In [ ]:
show("bruisekit/samprobe.py", r"^class SamViTProbe")
show("bruisekit/samprobe.py", r"def resample_pos_embed_2d")

### What does the training loop look like?

One driver for every model in the study. A bespoke loop for any arm would make its numbers unreadable against the rest. It is idempotent and resumable — see the RESUME CONTRACT in the docstring.

In [ ]:
show("bruisekit/engine.py", r"def train_run")

### What learning rate, and why two of them?

6e-5 for the pretrained backbone, 6e-4 for the randomly-initialised head. Group membership is decided by `id()`, not by parameter name — a name-prefix rule would put every YOLO parameter in the wrong group.

In [ ]:
show("bruisekit/models.py", r"def build_param_groups")
show("bruisekit/engine.py", r"def lr_multiplier")

### How do you choose the decision threshold?

Swept over 481 cuts on **validation** and applied once to test. It is **not** the argmax: these sweeps are flat, so every cut within one standard error of the peak is statistically tied, and the tie is broken by **lowest complete-miss rate**.

In [ ]:
show("bruisekit/sweep.py", r"def sweep_cuts")
show("bruisekit/sweep.py", r"def select_cut")

### How do you count a 'complete miss', and why three columns?

`dice == 0` is the union of two clinically different failures: the model found nothing, or it outlined the wrong place. `wrong_place` is **derived** so the three columns cannot fail to add up, and the function raises if a table is internally inconsistent.

In [ ]:
show("bruisekit/itakd.py", r"def _miss_counts")

### How do you test fairness across skin tones?

Every gap is reported twice — overall and within the small-lesion stratum — because lesion size is confounded with skin tone in this test set. Cells with fewer than five patients get no confidence interval rather than a number nobody should trust.

In [ ]:
show("bruisekit/lesionsize.py", r"def assign_bins")
show("bruisekit/lesionsize.py", r"def fairness_conditioned")

### How does multi-teacher distillation work?

Two variants. Stage M routes **per image** on each teacher's soft Dice against the label; Stage O routes **per skin-tone group** on weights fitted once on validation. Both reduce exactly to the single-teacher loss when K = 1, which is what makes the contrast one-variable — and both are asserted to do so in `self_test`.

In [ ]:
show("bruisekit/multiteacher.py", r"def _build_routed_loss_class")
show("bruisekit/itakd.py", r"def _build_group_loss_class")

### How does the model know which skin-tone group an image is in?

`engine.train_run` iterates `(x, y, _)` and **discards the stem**, so the loss has no idea which images it is looking at. Rather than edit the shared training loop, the training loader is wrapped to record each batch's group indices. The loss **raises** if that record is missing — a silent fallback to uniform weights would turn the arm into a plain ensemble and report a plausible number for a different experiment.

In [ ]:
show("bruisekit/itakd.py", r"^class _GroupTaggingLoader")
show("bruisekit/itakd.py", r"def install_group_shim")

### How do you decide whether an experiment is worth running?

A pre-registered gate, computed on **validation only**, written to disk before anything touches test. Stage O's adds a clause the earlier gates lacked: it refuses unless the thing being routed on is actually estimable.

In [ ]:
show("bruisekit/itakd.py", r"def ita_group_gate")
show("bruisekit/itakd.py", r"def identifiability")

### How do you know the numbers reproduce?

Every model can be re-scored from its checkpoint and compared against the table its original run wrote. Note `resolve_runs`: the best seed is **not** the same for every model, and scoring one at another's best seed shows per-image gaps up to 0.49 Dice while looking exactly like a broken pipeline.

In [ ]:
show("bruisekit/inference.py", r"def resolve_runs")
show("bruisekit/inference.py", r"def reconcile")

### How does an experiment change behaviour without editing the pipeline?

It patches a name. Every shim falls through to whatever was bound before it when its arm is not active, so a session that trains several stages keeps them separate. Note that `build_model` is rebound in **two** modules — `engine` bound it by value at import, so patching one is not enough.

In [ ]:
show("bruisekit/samprobe.py", r"def install_n4_shim")
show("bruisekit/itakd.py", r"def install_loss_shim")

---

## Three rules that explain most of what you just read

1. **Nothing is fitted on test.** Thresholds, best seeds and every gate are
   decided on validation, and a gate's verdict is written to disk before any test
   pass runs.
2. **An experiment patches a name; it never edits the pipeline.** Stage modules
   are deliberately absent from the bundle build's copy list, so an experiment
   that returns nothing cannot become a dependency of the code that produces the
   main results.
3. **A function that cannot do its job raises.** It does not warn and it does not
   return a plausible default. Most `raise` statements in this codebase mark a
   place where a silent fallback once produced a believable wrong number.

## If a cell says "not found"

The code was renamed. Regenerate this notebook —
`python scripts/89_generate_code_tour_notebook.py` — which **fails at build time**
on a stale anchor rather than emitting cells that break in front of an audience.

A file-level index of the same material, with line numbers resolved at build
time, is in **`docs/CODE_MAP.md`**.